# Rehabilitation Assessment System
## Activity Classifier + ROM-Based Rehab Scoring

**Pipeline:**
```
Raw sensor data
      ↓
Step 1 — Unit conversion (correct physical units)
      ↓
Step 2 — Wavelet denoising
      ↓
Step 3 — Complementary filter → knee angles
      ↓
Step 4 — LSTM Classifier → activity per window (smoothed)
      ↓
Step 5 — ROM per predicted activity
      ↓
Step 6 — Compare to healthy baseline → rehab progress %
      ↓
Step 7 — Rehabilitation report
```

**Important notes:**
- Classifier input uses data divided by 32768 only (matches training)
- Knee angle uses correct physical units (gyro x2000, accel x2)
- Smoothing removes boundary confusion between activities
- Progress % is relative comparison — same method for patient and reference

## CELL 1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pywt
import joblib
import json
import warnings
from collections import Counter
warnings.filterwarnings('ignore')

import tensorflow as tf

print('TensorFlow version:', tf.__version__)
print('All imports done.')

## CELL 2 — Mount Drive & Load Models

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# -------------------------------------------------------
BASE_PATH    = '/content/drive/MyDrive/Activity_demo/'
MODEL_PATH   = BASE_PATH + 'models/lstm_classifier.keras'
SCALER_PATH  = BASE_PATH + 'models/scaler.pkl'
META_PATH    = BASE_PATH + 'models/model_meta.json'
REHAB_PATH   = BASE_PATH + 'healthy_baseline.csv'
PATIENT_PATH = BASE_PATH + 
# -------------------------------------------------------

# Load classifier
print('Loading LSTM classifier...')
classifier = tf.keras.models.load_model(MODEL_PATH)
print('Classifier loaded.')

# Load scaler
scaler = joblib.load(SCALER_PATH)
print('Scaler loaded.')

# Load activity names from metadata
with open(META_PATH, 'r') as f:
    meta = json.load(f)
ACTIVITY_NAMES = meta['activity_names']
print(f'Activity classes: {ACTIVITY_NAMES}')

# Load healthy ROM baseline — subjects 1-14 average per activity
rehab_rules = pd.read_csv(REHAB_PATH)
healthy_baseline = (
    rehab_rules[rehab_rules['subject_id'] <= 14]
    .groupby('activity')['ROM_avg']
    .mean()
    .round(4)
)
print(f'\nHealthy ROM baseline (degrees):')
print(healthy_baseline.to_string())

# Load patient raw data
print(f'\nLoading patient data...')
patient_raw = pd.read_csv(PATIENT_PATH)
if 'Unnamed: 0' in patient_raw.columns:
    patient_raw = patient_raw.drop(columns=['Unnamed: 0'])
print(f'Shape: {patient_raw.shape}')
print(f'Activities in file: {patient_raw["activity"].value_counts().to_dict()}')

## CELL 3 — Create Two Preprocessed Copies

**Why two copies?**
- The classifier was trained on data divided by 32768 only
- The knee angle needs correct physical units (gyro x2000, accel x2)
- We must give the classifier the same scale it saw during training

In [ ]:
# Identify column groups
gyro_cols       = [c for c in patient_raw.columns if 'gyroscope' in c]
accel_cols      = [c for c in patient_raw.columns if 'accelerometer' in c]
emg_cols        = ['EMG_right', 'EMG_left']
all_sensor_cols = gyro_cols + accel_cols

print(f'Gyro columns  : {len(gyro_cols)}')
print(f'Accel columns : {len(accel_cols)}')

# Wavelet denoising function
def wavelet_denoise(signal, wavelet='db4', level=4):
    coeffs    = pywt.wavedec(signal, wavelet, level=level)
    sigma     = np.median(np.abs(coeffs[-1])) / 0.6745
    threshold = sigma * np.sqrt(2 * np.log(len(signal)))
    coeffs_t  = [coeffs[0]] + [
        pywt.threshold(d, threshold, mode='soft') for d in coeffs[1:]
    ]
    return pywt.waverec(coeffs_t, wavelet)[:len(signal)]

# --- COPY 1: For CLASSIFIER ---
# Must match training preprocessing: divide by 32768 only
print('\nCreating classifier copy (divide by 32768)...')
df_clf = patient_raw.copy()
df_clf[all_sensor_cols] = df_clf[all_sensor_cols] / 32768.0
for col in all_sensor_cols:
    df_clf[col] = wavelet_denoise(df_clf[col].values)
print('Classifier copy ready.')

# --- COPY 2: For KNEE ANGLE CALCULATION ---
# Correct physical units:
#   gyroscope:     raw / 32768 * 2000 → degrees/sec
#   accelerometer: raw / 32768 * 2    → g
print('Creating angle copy (correct physical units)...')
df_ang = patient_raw.copy()
df_ang[gyro_cols]  = df_ang[gyro_cols]  / 32768.0 * 2000
df_ang[accel_cols] = df_ang[accel_cols] / 32768.0 * 2
for col in all_sensor_cols:
    df_ang[col] = wavelet_denoise(df_ang[col].values)
print('Angle copy ready.')
print(f'Gyro range: {df_ang[gyro_cols].values.min():.1f} to {df_ang[gyro_cols].values.max():.1f} deg/s')
print(f'Accel range: {df_ang[accel_cols].values.min():.3f} to {df_ang[accel_cols].values.max():.3f} g')

## CELL 4 — Complementary Filter → Knee Angles

In [ ]:
ALPHA = 0.98  # 98% gyroscope + 2% accelerometer
DT    = 0.01  # 100Hz sampling rate

def complementary_filter(gyro, accel_x, accel_z, alpha=ALPHA, dt=DT):
    """
    Computes orientation angle by blending:
    - 98% gyroscope (accurate short-term, drifts long-term)
    - 2% accelerometer (corrects drift using gravity direction)
    """
    angles = np.zeros(len(gyro))
    angle  = 0.0
    for i in range(1, len(gyro)):
        accel_angle = np.degrees(np.arctan2(accel_x[i], accel_z[i]))
        angle       = alpha * (angle + gyro[i] * dt) + (1 - alpha) * accel_angle
        angles[i]   = angle
    return angles

# Subject 17 uses RIGHT side sensors
# (subjects 3,4,5,7 have corrupted right side — use left instead)
thigh_angle = complementary_filter(
    df_ang['gyroscope_right_thigh_x'].values,
    df_ang['accelerometer_right_thigh_x'].values,
    df_ang['accelerometer_right_thigh_z'].values
)
shin_angle = complementary_filter(
    df_ang['gyroscope_right_shin_x'].values,
    df_ang['accelerometer_right_shin_x'].values,
    df_ang['accelerometer_right_shin_z'].values
)

# Knee angle = relative angle between thigh and shin
knee_angle = thigh_angle - shin_angle

# Add knee angle to classifier dataframe for later ROM calculation
df_clf['knee_angle'] = knee_angle

print('Knee angles computed.')
print(f'Full trial range: {knee_angle.min():.2f} to {knee_angle.max():.2f} degrees')
print(f'Full trial ROM  : {knee_angle.max() - knee_angle.min():.2f} degrees')

## CELL 5 — LSTM Classifier with Smoothing

**Smoothing:** For each window, look at 10 surrounding windows
and take the majority vote. Fixes boundary confusion.

**Threshold:** Only keep activities detected in 20+ windows (~2.5 sec).
Removes spurious single-window predictions.

In [ ]:
WINDOW_SIZE             = 50   # 0.5 seconds at 100Hz
STEP_SIZE               = 25   # 50% overlap
SMOOTHING_WINDOW        = 10   # majority vote over ±10 windows
MIN_CONSECUTIVE_WINDOWS = 20   # minimum windows to accept an activity

# --- Feature columns (must match training exactly) ---
feature_cols = accel_cols + gyro_cols + emg_cols
X_patient    = df_clf[feature_cols].values
X_scaled     = scaler.transform(X_patient)

# --- Create sliding windows ---
X_windows, window_indices = [], []
for i in range(0, len(X_scaled) - WINDOW_SIZE, STEP_SIZE):
    X_windows.append(X_scaled[i : i + WINDOW_SIZE])
    window_indices.append(i + WINDOW_SIZE - 1)

X_windows = np.array(X_windows, dtype=np.float32)
print(f'Windows created: {X_windows.shape}')

# --- Run LSTM classifier ---
print('Running classifier...')
proba  = classifier.predict(X_windows, batch_size=256, verbose=0)
preds  = np.argmax(proba, axis=1)
# Convert to plain Python strings (important — avoids np.str_ issues)
labels = [str(ACTIVITY_NAMES[p]) for p in preds]

print(f'\nBefore smoothing:')
print(Counter(labels))

# --- Smoothing: majority vote over surrounding windows ---
def smooth_predictions(predictions, window=SMOOTHING_WINDOW):
    smoothed = []
    pred_arr = np.array(predictions)
    for i in range(len(pred_arr)):
        start    = max(0, i - window)
        end      = min(len(pred_arr), i + window)
        chunk    = pred_arr[start:end]
        unique, counts = np.unique(chunk, return_counts=True)
        majority = str(unique[np.argmax(counts)])
        smoothed.append(majority)
    return smoothed

labels_smoothed = smooth_predictions(labels)
print(f'\nAfter smoothing:')
print(Counter(labels_smoothed))

# --- Minimum threshold ---
activity_counts  = Counter(labels_smoothed)
valid_activities = {
    str(act) for act, count in activity_counts.items()
    if count >= MIN_CONSECUTIVE_WINDOWS
}
print(f'\nActivities passing threshold ({MIN_CONSECUTIVE_WINDOWS} windows):')
print(valid_activities)

# --- Map predictions back to rows ---
df_clf['predicted_activity'] = np.nan
for idx, label in zip(window_indices, labels_smoothed):
    if str(label) in valid_activities:
        df_clf.loc[idx, 'predicted_activity'] = str(label)

df_clf['predicted_activity'] = (
    df_clf['predicted_activity']
    .ffill().bfill()
)

print(f'\nFinal predictions per row:')
print(df_clf['predicted_activity'].value_counts())
print(f'\nActual activities in file:')
print(patient_raw['activity'].value_counts())

## CELL 6 — ROM Per Predicted Activity

In [ ]:
REHAB_ACTIVITIES = [
    'walking', 'going_up', 'going_down',
    'running', 'sitting', 'sitting_down',
    'standing', 'standing_up'
]

rom_results = []

for activity in REHAB_ACTIVITIES:
    activity_rows = df_clf[
        df_clf['predicted_activity'] == activity
    ]

    if len(activity_rows) < 50:
        print(f'  Skipping {activity}: only {len(activity_rows)} rows')
        continue

    ROM = activity_rows['knee_angle'].max() - activity_rows['knee_angle'].min()

    if activity not in healthy_baseline.index:
        print(f'  Skipping {activity}: no healthy baseline')
        continue

    healthy_avg = float(healthy_baseline[activity])
    progress    = min((ROM / healthy_avg) * 100, 100.0)

    rom_results.append({
        'activity'     : activity,
        'patient_ROM'  : round(float(ROM), 2),
        'healthy_avg'  : round(healthy_avg, 2),
        'progress_pct' : round(progress, 1),
        'rows_detected': len(activity_rows)
    })

results_df = pd.DataFrame(rom_results)
print('\nROM results:')
print(results_df.to_string(index=False))

## CELL 7 — Rehabilitation Report

In [ ]:
print('=' * 58)
print('         REHABILITATION ASSESSMENT REPORT')
print('=' * 58)
print(f'Patient     : Subject 17 (simulated patient)')
print(f'Classifier  : LSTM (Test Accuracy: {meta["test_accuracy"]*100:.1f}%)')
print(f'Reference   : HuGaDB healthy subjects (subjects 1-14)')
print('=' * 58)

if len(results_df) == 0:
    print('No activities detected with sufficient data.')
    print('Patient may need to perform activities for longer.')
else:
    for _, row in results_df.iterrows():
        filled = int(row['progress_pct'] / 5)
        empty  = 20 - filled
        bar    = '█' * filled + '░' * empty

        print(f"\n {row['activity'].upper().replace('_', ' ')}")
        print(f"  Patient ROM   : {row['patient_ROM']:.1f}°")
        print(f"  Healthy avg   : {row['healthy_avg']:.1f}°")
        print(f"  Progress      : [{bar}] {row['progress_pct']:.1f}%")
        print(f"  Data detected : {row['rows_detected']} rows")

    overall = results_df['progress_pct'].mean()
    print(f'\n{"=" * 58}')
    print(f'  OVERALL PROGRESS SCORE : {overall:.1f}%')
    print(f'{"=" * 58}')
    print('  Note: Progress % = patient ROM / healthy average ROM x 100')
    print('  Same IMU method used for both patient and reference.')
    print('=' * 58)

## CELL 8 — Visualise Results

In [ ]:
if len(results_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # ROM comparison bar chart
    x     = np.arange(len(results_df))
    width = 0.35
    axes[0].bar(x - width/2, results_df['patient_ROM'],
                width, label='Patient ROM', color='steelblue', alpha=0.8)
    axes[0].bar(x + width/2, results_df['healthy_avg'],
                width, label='Healthy Average', color='lightgreen', alpha=0.8)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(
        [a.replace('_', '\n') for a in results_df['activity']], fontsize=9)
    axes[0].set_ylabel('Knee ROM (degrees)')
    axes[0].set_title('Patient ROM vs Healthy Average')
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)

    # Progress % horizontal bar chart
    colors = [
        'green'  if p >= 90 else
        'orange' if p >= 70 else
        'red'
        for p in results_df['progress_pct']
    ]
    axes[1].barh(
        results_df['activity'].str.replace('_', ' '),
        results_df['progress_pct'],
        color=colors, alpha=0.8
    )
    axes[1].axvline(x=100, color='black', linestyle='--',
                    linewidth=1.5, label='Healthy = 100%')
    axes[1].set_xlabel('Progress (%)')
    axes[1].set_title('Rehabilitation Progress per Activity')
    axes[1].set_xlim(0, 115)
    axes[1].grid(axis='x', alpha=0.3)

    patches = [
        mpatches.Patch(color='green',  label='>= 90%  Near healthy'),
        mpatches.Patch(color='orange', label='70-89%  Reduced'),
        mpatches.Patch(color='red',    label='< 70%   Significantly reduced'),
    ]
    axes[1].legend(handles=patches, loc='lower right', fontsize=8)

    plt.suptitle(
        f'Rehabilitation Assessment — Subject 17 | Overall: {results_df["progress_pct"].mean():.1f}%',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(BASE_PATH + 'rehab_report.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Report chart saved to Google Drive.')

## CELL 9 — Classifier Accuracy vs Ground Truth

In [ ]:
from sklearn.metrics import classification_report

eval_df = df_clf.copy()
eval_df['true_activity'] = patient_raw['activity'].values

# Only evaluate rows where both true and predicted are in our 8 classes
valid = set(ACTIVITY_NAMES)
eval_df = eval_df[
    eval_df['true_activity'].isin(valid) &
    eval_df['predicted_activity'].isin(valid)
]

if len(eval_df) > 0:
    print('Classifier performance on this patient trial:')
    print(classification_report(
        eval_df['true_activity'],
        eval_df['predicted_activity'],
        labels=ACTIVITY_NAMES,
        target_names=ACTIVITY_NAMES,
        zero_division=0
    ))
else:
    print('No rows with matching activity labels to evaluate.')